In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
!pip install --no-deps xformers --index-url https://download.pytorch.org/whl/cu121
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install "trl==0.24.0" "peft" "accelerate" "bitsandbytes"

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 83.6 MB/s eta 0:00:00:00:0100:01
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-plnvuj2b/unsloth_bf3052fd31584e16995905e56862b45b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-plnvuj2b/unsloth_bf3052fd31584e16995905e56862b45b
  Resolved https://github.com/unslothai/unsloth.git to commit 12f8525bc6f04815b90f45242f14498d83dea4ec
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.6 MB/s eta 0:00:00:

In [4]:
from unsloth import FastLanguageModel
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Загрузка модели
max_seq_length = 2048 
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16, lora_dropout = 0, bias = "none", use_gradient_checkpointing = "unsloth", random_state = 3407,
)

# 2. Подготовка датасета
data = [
    {"instruction": "Исправь синтаксис Python", "input": "def hello() print('hi')", "output": "def hello():\n    print('hi')"},
    {"instruction": "Исправь синтаксис Python", "input": "for i in range(10) print(i)", "output": "for i in range(10):\n    print(i)"},
    {"instruction": "Убери лишнее, оставь суть", "input": "Привет! Меня зовут Вася, я хочу сказать, что сервер упал из-за ошибки 500.", "output": "Статус: Сервер недоступен. Ошибка: 500. Пользователь: Вася."},
    {"instruction": "Исправь синтаксис Python", "input": "if x = 5: return True", "output": "if x == 5:\n    return True"}
]
data = data * 50 
dataset = Dataset.from_list(data)

vat_brain_prompt = """Ниже приведена инструкция и входные данные. Выдай ТОЛЬКО исправленный результат, без приветствий, объяснений и markdown-разметки.

### Инструкция:
{}

### Входные данные:
{}

### Результат:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = vat_brain_prompt.format(inst, inp, out) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

mapped_dataset = dataset.map(formatting_prompts_func, batched = True,)

# 3. Обучение
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer, 
    train_dataset = mapped_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, 
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
10,1.578908
20,0.221990
30,0.027578
40,0.015987
50,0.016814
60,0.016661


In [5]:
# 1. Включаем режим быстрой генерации (в 2 раза быстрее)
FastLanguageModel.for_inference(model)

# 2. Формируем запрос с ошибкой (забыли двоеточие в функции)
inputs = tokenizer(
[
    vat_brain_prompt.format(
        "Исправь синтаксис Python", 
        "def calculate_sum(a, b) return a+b", # Вход с ошибкой
        "", # Оставляем пустым, чтобы модель сама дописала ответ
    )
], return_tensors = "pt").to("cuda")

# 3. Генерируем ответ
outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

print("\n=== ОТВЕТ 'АКВАРИУМА' ===")
# Отрезаем промпт и показываем только чистый результат
print(response.split("### Результат:\n")[1])

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 


=== ОТВЕТ 'АКВАРИУМА' ===
def calculate_sum(a, b):
    return a + b
